In [1]:
import sys; sys.path.append("..")
import shutil, joblib
import numpy as np
import pandas as pd
from src import gbm

df = pd.read_csv("../data/processed/policies_clean.csv")
tr_ids = pd.read_csv("../data/processed/train_ids.csv")["IDpol"]
train = df[df["IDpol"].isin(tr_ids)].reset_index(drop=True)

shutil.copy("../models/gbm_v2.joblib", "../models/gbm_v2_backup.joblib")
b = joblib.load("../models/gbm_v2.joblib")

rate = b["model"].predict(gbm.make_X(train.assign(Exposure=1.0), b["levels"], b["features"]))
b["reference_annual_rate"] = float(rate.mean())
b["tier_edges"] = np.quantile(rate, np.arange(0.1, 1.0, 0.1)).tolist()      # 9 cut points -> 10 tiers
b["ranges"] = {c: (int(train[c].min()), int(train[c].max()))
               for c in ["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]}
joblib.dump(b, "../models/gbm_v2.joblib")
print("reference annual rate:", round(b["reference_annual_rate"], 4))
print("tier edges:", np.round(b["tier_edges"], 4))
print("ranges:", b["ranges"])

reference annual rate: 0.0811
tier edges: [0.0354 0.0432 0.0496 0.0557 0.0619 0.0689 0.0782 0.0961 0.1445]
ranges: {'VehPower': (4, 15), 'VehAge': (0, 100), 'DrivAge': (18, 100), 'BonusMalus': (50, 230), 'Density': (1, 27000)}
